# 01 — Residue Classes mod 6

Purpose: first measurable constraint result.

\[
p > 3 \Rightarrow p \equiv \pm 1 \pmod{6}
\]

Prime residues remain under constraint in 1,5 mod 6; others drift.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, math, zipfile

NOTEBOOK_ID = "01_residue_classes_mod6"
NOTEBOOK_NUM = NOTEBOOK_ID.split("_")[0]

OUT = Path(NOTEBOOK_ID)
DATA_DIR = OUT/"data"
DOCS_DIR = OUT/"docs"
FIG_DIR = OUT/"figures"
TEX_DIR = OUT/"tex"

for d in [DATA_DIR, DOCS_DIR, FIG_DIR, TEX_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
def sieve(n):
    s = np.ones(n,dtype=bool)
    s[:2]=False
    for i in range(2,int(math.sqrt(n))+1):
        if s[i]:
            s[i*i:n:i]=False
    return np.nonzero(s)[0]

N_MAX=200000
primes=sieve(N_MAX)
p=primes[primes>3]

In [ ]:
def counts(vals):
    c=np.bincount(vals%6,minlength=6)
    return pd.DataFrame({"r":range(6),"count":c,"share":c/c.sum()})

int_df=counts(np.arange(1,N_MAX))
prime_df=counts(p)

df=int_df.merge(prime_df,on="r",suffixes=("_int","_prime"))
df

In [ ]:
valid = np.isin(p%6,[1,5])
cgcs = valid.mean()
drift = 1-cgcs

measurement = {
    "cgcs":float(cgcs),
    "drift":float(drift),
    "count":int(len(p))
}
measurement

In [ ]:
import matplotlib.pyplot as plt

fig,ax=plt.subplots()
ax.bar(df["r"]-0.2,df["share_int"],0.4,label="integers")
ax.bar(df["r"]+0.2,df["share_prime"],0.4,label="primes>3")
ax.legend()
ax.set_title("mod6 distribution")

fig_path=FIG_DIR/f"{NOTEBOOK_NUM}_mod6.png"
fig.savefig(fig_path)
fig_path

In [ ]:
# exports
summary_path=DATA_DIR/f"{NOTEBOOK_NUM}_summary.csv"
df_path=DATA_DIR/f"{NOTEBOOK_NUM}_residues.csv"
meta_path=DATA_DIR/f"{NOTEBOOK_NUM}_metadata.json"

pd.DataFrame([measurement]).to_csv(summary_path,index=False)
df.to_csv(df_path,index=False)

import json
meta_path.write_text(json.dumps(measurement,indent=2))

interp_path=DOCS_DIR/f"{NOTEBOOK_NUM}_interpretation.md"
interp_path.write_text(f"CGCS={cgcs:.3f}, drift={drift:.3f}")

design_path=DOCS_DIR/f"{NOTEBOOK_NUM}_design_notes.md"
design_path.write_text("Residue constraint baseline notebook.")

summary_tex=TEX_DIR/f"{NOTEBOOK_NUM}_summary_snippet.tex"
summary_tex.write_text(f"CGCS={cgcs:.6f}, drift={drift:.6f}")

math_tex=TEX_DIR/f"{NOTEBOOK_NUM}_math_notes.tex"
math_tex.write_text(r"""\documentclass{article}
\usepackage{amsmath}
\begin{document}
\[
p>3 \Rightarrow p\equiv \pm1 \pmod{6}
\]
\end{document}
""")

summary_path, df_path, fig_path

In [ ]:
# export zip (pi-stage style)
EXPORT_NAME=f"{NOTEBOOK_ID}_export.zip"

with zipfile.ZipFile(EXPORT_NAME,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in [DOCS_DIR,DATA_DIR,FIG_DIR,TEX_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path,path.as_posix())

print(f"Export ready: {EXPORT_NAME}")
print("Tip: uncomment Colab lines below to download.")

# from google.colab import files
# files.download(EXPORT_NAME)